In [52]:
import torch
from torch import nn
from d2l import torch as d2l

In [53]:
# Query를 여러 Head로 분리

batch_size = 2
num_queries = 4

num_hiddens = 100
num_heads = 5
    
if num_hiddens % num_heads != 0:
    raise ValueError(
        "num_hiddens must be divisible by num_heads."
    )    
    
# 100 // 5 = 20
head_dim = (
    num_hiddens // num_heads
)

# Queries:
# [B, Q, H] = [2, 4, 100]
queries = torch.ones(
    (
        batch_size,
        num_queries,
        num_hiddens,
    )
)

# 하나의 Linear Layer를 계산한 뒤, 결과를 5개의 Head로 분리
W_q = nn.LazyLinear(
    out_features=num_hiddens,
    bias=False,
)

# Projected Queries:
# [B, Q, H] = [2, 4, 100]
projected_queries = W_q(
    queries
)

# Hidden Dimension 100을 Head 5개 × Head Dimension 20으로 분리:
# [B, Q, H] -> [B, Q, Head, Head Dimension] = [2, 4, 5, 20]
split_queries = projected_queries.reshape(
    batch_size,
    num_queries,
    num_heads,
    head_dim,
)

# [B, Q, Head, D_head] -> [B, Head, Q, D_head] = [2, 5, 4, 20]
head_first_queries = split_queries.permute(
    0,
    2,
    1,
    3,
)

# [B, Head, Q, D_head] -> [B × Head, Q, D_head] = [10, 4, 20]
parallel_queries = (
    head_first_queries.reshape(
        batch_size * num_heads,
        num_queries,
        head_dim,
    )
)

print(
    "Projected Queries:",
    tuple(projected_queries.shape),
)

print(
    "Split Queries:",
    tuple(split_queries.shape),
)

print(
    "Head-first Queries:",
    tuple(head_first_queries.shape),
)

print(
    "Parallel Queries:",
    tuple(parallel_queries.shape),
)

print(
    "Dimension per Head:",
    head_dim,
)

Projected Queries: (2, 4, 100)
Split Queries: (2, 4, 5, 20)
Head-first Queries: (2, 5, 4, 20)
Parallel Queries: (10, 4, 20)
Dimension per Head: 20


In [54]:
# Queries, Keys, Values를 여러 Head로 분리

# [B, T, H] -> [B × Head, T, D_head]
def transpose_qkv(
    X: torch.Tensor, 
    num_heads: int,
) -> torch.Tensor:
    
    (
        batch_size,
        num_steps,
        num_hiddens,
    ) = X.shape # [B, T, H]
    
    if num_hiddens % num_heads != 0:
        raise ValueError(
            "num_hiddens must be divisible by num_heads."
        )
    
    head_dim = (
        num_hiddens // num_heads
    )
    
    # [B, T, H] -> [B, T, Head, D_head]
    X = X.reshape(
        batch_size,
        num_steps,
        num_heads,
        head_dim,
    )
    
    # [B, T, Head, D_head] -> [B, Head, T, D_head]
    X = X.permute(
        0,
        2,
        1,
        3,
    )
    
    # Batch와 Head를 결합:
    # [B, Head, T, D_head] -> [B × Head, T, D_head]
    return X.reshape(
        batch_size * num_heads,
        num_steps,
        head_dim,
    )

In [55]:
# Cell 2의 Projected Query reusing:
# [2, 4, 100] -> [2 × 5, 4, 100 / 5] -> [10, 4, 20]
transposed_queries = transpose_qkv(
    X=projected_queries,
    num_heads=num_heads,
)


print(
    "Before:",
    tuple(projected_queries.shape),
)

print(
    "After:",
    tuple(transposed_queries.shape),
)

d2l.check_shape(
    transposed_queries,
    (
        batch_size * num_heads,
        num_queries,
        head_dim,
    ),
)

torch.testing.assert_close(
    transposed_queries,
    parallel_queries,
)

Before: (2, 4, 100)
After: (10, 4, 20)


In [56]:
# 여러 Head의 Output을 다시 결합

# [B × Head, T, D_head] -> [B, T, H]
def transpose_output(
    X: torch.Tensor,
    num_heads: int,
) -> torch.Tensor:

    (
        batch_times_heads,
        num_steps,
        head_dim,
    ) = X.shape # [B × Head, T, D_head]
    
    if batch_times_heads % num_heads != 0:
        raise ValueError(
            "The leading dimension must be divisible by num_heads."
        )
    
    batch_size = (
        batch_times_heads // num_heads
    )
    
    # Batch와 Head를 다시 분리
    # [B × Head, T, D_head] -> [B, Head, T, D_head]
    X = X.reshape(
        batch_size,
        num_heads,
        num_steps,
        head_dim,
    )
    
    # Original Sequence 순서로 복원
    # [B, Head, T, D_head] -> [B, T, Head, D_head]
    X = X.permute(
        0,
        2,
        1,
        3,
    )

    # Head와 Head Dimension을 결합
    # [B, T, Head, D_head] -> [B, T, H]
    return X.reshape(
        batch_size,
        num_steps,
        num_heads * head_dim,
    )

In [57]:
# [B, T(Q), H] -> [B × Head, T(Q), D_head]
# [2, 4, 100]  -> [10, 4, 20]
transposed_queries = transpose_qkv(
    X=projected_queries,
    num_heads=num_heads,
)


# [B × Head, Q, D_head] -> [B, Q, H]
# [10, 4, 20] -> [2, 4, 100]
restored_queries = transpose_output(
    X=transposed_queries,
    num_heads=num_heads,
)


print(
    "Transposed Queries:",
    tuple(transposed_queries.shape),
)

print(
    "Restored Queries:",
    tuple(restored_queries.shape),
)


d2l.check_shape(
    restored_queries,
    (
        batch_size,
        num_queries,
        num_hiddens,
    ),
)


# transpose_output이 transpose_qkv의
# 정확한 역연산인지 tensor 값까지 검증
torch.testing.assert_close(
    restored_queries,
    projected_queries,
)

Transposed Queries: (10, 4, 20)
Restored Queries: (2, 4, 100)


In [58]:
# Multi-Head Attention

class MultiHeadAttention(
    d2l.Module
):
    
    def __init__(
        self,
        num_hiddens: int,
        num_heads: int,
        dropout: float,
        bias: bool = False,
    ) -> None:
        super().__init__()
        
        if num_hiddens % num_heads != 0:
            raise ValueError(
                "num_hiddens must be divisible by num_heads."
            )

        self.num_heads = num_heads

        # 각 Head는 Scaled Dot-Product Attention 사용
        self.attention = d2l.DotProductAttention(
            dropout=dropout,
        )
        
        
        # Queries, Keys, Values를 서로 다른
        # representation subspace로 Projection
        #
        # Output: [B, T, H]
        self.W_q = nn.LazyLinear(
            out_features=num_hiddens,
            bias=bias,
        )

        self.W_k = nn.LazyLinear(
            out_features=num_hiddens,
            bias=bias,
        )

        self.W_v = nn.LazyLinear(
            out_features=num_hiddens,
            bias=bias,
        )

        
        # 여러 Head를 결합한 뒤 적용하는
        # 최종 Output Projection
        self.W_o = nn.LazyLinear(
            out_features=num_hiddens,
            bias=bias,
        )
        
      
    def forward(
        self,
        queries: torch.Tensor,
        keys: torch.Tensor,
        values: torch.Tensor,
        valid_lens: torch.Tensor | None,
    ) -> torch.Tensor:  
        
        # Original shape:
        #
        # Queries: [B, Q, D_q]
        # Keys   : [B, K, D_k]
        # Values : [B, K, D_v] 
        
        
        # Linear Projection 후 Head 분리:
        #
        # [B, Q, D_q] -> [B, Q, H] -> [B × Head, Q, D_head]
        queries = transpose_qkv(
            X=self.W_q(queries),
            num_heads=self.num_heads,
        )

        # [B, K, D_k] -> [B, K, H] -> [B × Head, K, D_head]
        keys = transpose_qkv(
            X=self.W_k(keys),
            num_heads=self.num_heads,
        )
        
        # [B, K, D_v] -> [B, K, H] -> [B × Head, K, D_head]
        values = transpose_qkv(
            X=self.W_v(values),
            num_heads=self.num_heads,
        )
        
        # valid length를 모든 Head에 동일하게 적용: [B × Head]   
        if valid_lens is None:
            repeated_valid_lens = None

        else:
            repeated_valid_lens = (
                torch.repeat_interleave(
                    valid_lens,
                    repeats=self.num_heads, # H
                    dim=0,
                )
            ) 
            
        # 모든 Head를 하나의 큰 Batch처럼 병렬 계산:
        #  Parallel Scaled Dot-Product Attention
        #
        # Queries : [B × Head, Q, D_head]
        # Keys    : [B × Head, K, D_head]
        # Values  : [B × Head, K, D_head]
        #
        # Output  : [B × Head, Q, D_head]
        output = self.attention(
            queries,
            keys,
            values,
            repeated_valid_lens,
        )

        # 여러 Head를 다시 결합:
        # [B × Head, Q, D_head] -> [B, Q, H]
        output_concat = transpose_output(
            X=output,
            num_heads=self.num_heads,
        )

        # Head를 결합한 결과에 Output Projection 적용:
        # [B, Q, H] -> [B, Q, H]
        return self.W_o(
            output_concat
        )

In [59]:
# Multi-Head Attention Test

num_hiddens = 100
num_heads = 5

batch_size = 2
num_queries = 4
num_kv_pairs = 6


attention = MultiHeadAttention(
    num_hiddens=num_hiddens,
    num_heads=num_heads,
    dropout=0.5,
)


# Dropout 비활성화
attention.eval()


# Queries:
# [B, Q, H] = [2, 4, 100]
X = torch.ones(
    (
        batch_size,
        num_queries,
        num_hiddens,
    )
)


# Keys와 Values:
# [B, K, H] = [2, 6, 100]
Y = torch.ones(
    (
        batch_size,
        num_kv_pairs,
        num_hiddens,
    )
)

valid_lens = torch.tensor([
    3,
    2,
])

with torch.no_grad():
    output = attention(
        queries=X,
        keys=Y,
        values=Y,
        valid_lens=valid_lens,
    )

print(
    "Queries:",
    tuple(X.shape),
)

print(
    "Keys and Values:",
    tuple(Y.shape),
)

print(
    "Multi-Head Output:",
    tuple(output.shape),
)

# DotProductAttention 내부에 저장된 shape:
#
# [B × Head, Q, K]
raw_attention_weights = getattr(
    attention.attention,
    "attention_weights",
    None,
)

print(
    "Parallel Attention Weights:",
    tuple(raw_attention_weights.shape),
)


# Batch와 Head를 다시 분리
#
# [B × Head, Q, K]
#          ↓
# [B, Head, Q, K]
head_attention_weights = (
    raw_attention_weights.reshape(
        batch_size,
        num_heads,
        num_queries,
        num_kv_pairs,
    )
)


print(
    "Head Attention Weights:",
    tuple(head_attention_weights.shape),
)

Queries: (2, 4, 100)
Keys and Values: (2, 6, 100)
Multi-Head Output: (2, 4, 100)
Parallel Attention Weights: (10, 4, 6)
Head Attention Weights: (2, 5, 4, 6)
